In [11]:
import pandas as pd
from pathlib import Path
from typing import Union, List, Optional, Dict, Iterator
from datasets import Dataset, DatasetDict
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
import pandas as pd
from pathlib import Path
from typing import Union, List, Optional

def read_parquet_data_recursive(
    base_path: Union[str, Path],
    file_pattern: str = "*.parquet",
    columns: Optional[List[str]] = None,
    add_metadata: bool = True
) -> pd.DataFrame:
    """
    Recursively read all parquet files from a directory structure.
    
    Parameters:
    -----------
    base_path : str or Path
        Base directory to start recursive search
    file_pattern : str, default "*.parquet"
        Pattern to match files (e.g., "*.parquet", "*.csv")
    columns : list, optional
        Specific columns to read from parquet files
    add_metadata : bool, default True
        Whether to add metadata columns (directory, subdirectory, filename)
        
    Returns:
    --------
    pd.DataFrame
        Combined dataframe with all parquet data and metadata columns
    """
    base_path = Path(base_path)
    
    if not base_path.exists():
        raise ValueError(f"Path does not exist: {base_path}")
    
    # Recursively find all parquet files
    parquet_files = list(base_path.rglob(file_pattern))
    
    if not parquet_files:
        raise ValueError(f"No files matching '{file_pattern}' found in {base_path}")
    
    print(f"Found {len(parquet_files)} parquet files to process...")
    
    # Read and combine all parquet files
    dfs = []
    for parquet_file in parquet_files:
        try:
            # Read the parquet file
            df = pd.read_parquet(parquet_file, columns=columns)
            
            if add_metadata:
                # Get relative path from base directory
                relative_path = parquet_file.relative_to(base_path)
                
                # Extract directory information
                parts = relative_path.parts
                
                # Add metadata columns
                df['source_filename'] = parquet_file.name
                df['source_directory'] = parts[0] if len(parts) > 1 else ''
                df['source_subdirectory'] = parts[1] if len(parts) > 2 else ''
                df['source_full_path'] = str(relative_path)
                
            dfs.append(df)
            
        except Exception as e:
            print(f"Error reading {parquet_file}: {e}")
            continue
    
    if not dfs:
        raise ValueError("No parquet files could be successfully read")
    
    # Combine all dataframes
    combined_df = pd.concat(dfs, ignore_index=True)
    
    print(f"Successfully combined {len(dfs)} files into dataframe with {len(combined_df)} rows")
    
    return combined_df


def read_parquet_with_custom_metadata(
    base_path: Union[str, Path],
    max_depth: Optional[int] = None
) -> pd.DataFrame:
    """
    Advanced version that handles arbitrary directory depth.
    
    Parameters:
    -----------
    base_path : str or Path
        Base directory to start recursive search
    max_depth : int, optional
        Maximum directory depth to traverse (None for unlimited)
        
    Returns:
    --------
    pd.DataFrame
        Combined dataframe with flexible metadata columns
    """
    base_path = Path(base_path)
    parquet_files = list(base_path.rglob("*.parquet"))
    
    dfs = []
    for parquet_file in parquet_files:
        try:
            # Check depth limit
            relative_path = parquet_file.relative_to(base_path)
            depth = len(relative_path.parts) - 1  # Subtract filename
            
            if max_depth is not None and depth > max_depth:
                continue
            
            df = pd.read_parquet(parquet_file)
            
            # Add comprehensive metadata
            df['filename'] = parquet_file.name
            df['file_stem'] = parquet_file.stem  # Filename without extension
            
            # Add each directory level as separate column
            parts = relative_path.parent.parts
            for i, part in enumerate(parts):
                df[f'dir_level_{i}'] = part
            
            # Add full relative path
            df['full_relative_path'] = str(relative_path)
            df['depth_level'] = depth
            
            dfs.append(df)
            
        except Exception as e:
            print(f"Skipping {parquet_file}: {e}")
            continue
    
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


# Example usage for your data structure:
# # Basic usage - reads all parquet files recursively
# df = read_parquet_data_recursive('_provided/test_tracking')

# # Access metadata
# print(df[['source_directory', 'source_subdirectory', 'source_filename']].head())

# # Filter by specific category
# adaptable_snail_data = df[df['source_directory'] == 'AdaptableSnail']

# # Advanced usage with dynamic depth handling
# df_advanced = read_parquet_with_custom_metadata('_provided/test_tracking', max_depth=2)

# # See all unique directories
# print(df['source_directory'].unique())


In [3]:
train = pd.read_csv('data/MABe-mouse-behavior-detection/train.csv')
test = pd.read_csv('data/MABe-mouse-behavior-detection/test.csv')

In [4]:
train.head()

,lab_id,video_id,mouse1_strain,mouse1_color,mouse1_sex,mouse1_id,mouse1_age,mouse1_condition,mouse2_strain,mouse2_color,...,pix_per_cm_approx,video_width_pix,video_height_pix,arena_width_cm,arena_height_cm,arena_shape,arena_type,body_parts_tracked,behaviors_labeled,tracking_method
0,AdaptableSnail,44566106,CD-1 (ICR),white,male,10.0,8-12 weeks,wireless device,CD-1 (ICR),white,...,16.0,1228,1068,60.0,60.0,square,familiar,"[""body_center"", ""ear_left"", ""ear_right"", ""head...","[""mouse1,mouse2,approach"", ""mouse1,mouse2,atta...",DeepLabCut
1,AdaptableSnail,143861384,CD-1 (ICR),white,male,3.0,8-12 weeks,NaN,CD-1 (ICR),white,...,9.7,968,608,60.0,60.0,square,familiar,"[""body_center"", ""ear_left"", ""ear_right"", ""late...","[""mouse1,mouse2,approach"", ""mouse1,mouse2,atta...",DeepLabCut
2,AdaptableSnail,209576908,CD-1 (ICR),white,male,7.0,8-12 weeks,NaN,CD-1 (ICR),white,...,16.0,1266,1100,60.0,60.0,square,familiar,"[""body_center"", ""ear_left"", ""ear_right"", ""late...","[""mouse1,mouse2,approach"", ""mouse1,mouse2,atta...",DeepLabCut
3,AdaptableSnail,278643799,CD-1 (ICR),white,male,11.0,8-12 weeks,wireless device,CD-1 (ICR),white,...,16.0,1224,1100,60.0,60.0,square,familiar,"[""body_center"", ""ear_left"", ""ear_right"", ""head...","[""mouse1,mouse2,approach"", ""mouse1,mouse2,atta...",DeepLabCut
4,AdaptableSnail,351967631,CD-1 (ICR),white,male,14.0,8-12 weeks,NaN,CD-1 (ICR),white,...,16.0,1204,1068,60.0,60.0,square,familiar,"[""body_center"", ""ear_left"", ""ear_right"", ""late...","[""mouse1,mouse2,approach"", ""mouse1,mouse2,atta...",DeepLabCut


In [5]:
test.head()

,lab_id,video_id,mouse1_strain,mouse1_color,mouse1_sex,mouse1_id,mouse1_age,mouse1_condition,mouse2_strain,mouse2_color,...,pix_per_cm_approx,video_width_pix,video_height_pix,arena_width_cm,arena_height_cm,arena_shape,arena_type,body_parts_tracked,behaviors_labeled,tracking_method
0,AdaptableSnail,438887472,CD-1 (ICR),white,male,13.0,8-12 weeks,wireless device,CD-1 (ICR),white,...,16.0,1214,1090,60.0,60.0,square,familiar,"[""body_center"", ""ear_left"", ""ear_right"", ""head...","[""mouse1,mouse2,approach"", ""mouse1,mouse2,atta...",DeepLabCut


In [6]:
train.shape, test.shape

((8789, 38), (1, 38))

actual data itself

In [7]:
x = pd.read_parquet('data/MABe-mouse-behavior-detection/train_tracking/AdaptableSnail/44566106.parquet')
x.head()

,video_frame,mouse_id,bodypart,x,y
0,0,1,body_center,1161.543945,523.112976
1,0,1,ear_right,1146.305054,587.619995
2,0,1,headpiece_bottomfrontright,1163.192017,588.580017
3,0,1,headpiece_topbackright,1192.211060,558.434998
4,0,1,headpiece_topfrontleft,1191.343994,620.625977


In [8]:
x.shape

(1087658, 5)

In [9]:
y = pd.read_parquet('data/MABe-mouse-behavior-detection/train_annotation/AdaptableSnail/44566106.parquet')

In [10]:
y.shape

(342, 5)

# train annotation

In [ ]:
# Basic usage - reads all parquet files recursively
df_train_annotation = read_parquet_data_recursive('data/MABe-mouse-behavior-detection/train_tracking')
df_train_annotation.drop("source_subdirectory", axis=1, inplace=True)
df_train_annotation.head()

Found 8789 parquet files to process...


In [ ]:
ds_train_annotation = Dataset.from_pandas(df=df_train_annotation)
ds_train_annotation

In [ ]:
# ds_train_annotation.push_to_hub(repo_id="souriez/MABe-2025_train-annotation")

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 41.06ba/s]
Processing Files (1 / 1): 100%|██████████|  630kB /  630kB,  0.00B/s  
New Data Upload: 100%|██████████| 34.0kB / 34.0kB,  0.00B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.58 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/souriez/MABe-2025_train-annotation/commit/6ce63bfc3ee4dc2bb9014ec04d68224f2bfc040e', commit_message='Upload dataset', commit_description='', oid='6ce63bfc3ee4dc2bb9014ec04d68224f2bfc040e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/souriez/MABe-2025_train-annotation', endpoint='https://huggingface.co', repo_type='dataset', repo_id='souriez/MABe-2025_train-annotation'), pr_revision=None, pr_num=None)

In [ ]:
ds_train_annotation.save_to_disk('data/MABe-mouse-behavior-detection/hf/train_tracking/')

Saving the dataset (1/1 shards): 100%|██████████| 84066/84066 [00:00<00:00, 2006717.70 examples/s]


# test tracking

In [ ]:
df_test_tracking = read_parquet_data_recursive('data/MABe-mouse-behavior-detection/test_tracking')
df_test_tracking.drop("source_subdirectory", axis=1, inplace=True)
df_test_tracking.head()

Found 1 parquet files to process...
Successfully combined 1 files into dataframe with 1089866 rows


,video_frame,mouse_id,bodypart,x,y,source_filename,source_directory,source_full_path
0,0,1,body_center,909.577026,960.265991,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet
1,0,1,ear_left,861.718018,988.614014,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet
2,0,1,ear_right,882.953003,934.760010,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet
3,0,1,headpiece_bottombackright,828.593994,997.226013,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet
4,0,1,headpiece_bottomfrontleft,843.098022,948.182983,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet


In [ ]:
ds_test_tracking = Dataset.from_pandas(df=df_test_tracking)
ds_test_tracking

Dataset({
    features: ['video_frame', 'mouse_id', 'bodypart', 'x', 'y', 'source_filename', 'source_directory', 'source_full_path'],
    num_rows: 1089866
})

Processing Files (1 / 1): 100%|██████████| 10.6MB / 10.6MB, 8.80MB/s  

In [ ]:
# ds_test_tracking.push_to_hub("souriez/MABe-2025_test-tracking_RAW")

Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00,  6.09ba/s]
Processing Files (1 / 1): 100%|██████████| 10.6MB / 10.6MB, 7.54MB/s  
New Data Upload: 100%|██████████| 2.87MB / 2.87MB, 2.05MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.55s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/souriez/MABe-2025_test-tracking_RAW/commit/ed828054caf25401e76eb26a77f844ff9162062e', commit_message='Upload dataset', commit_description='', oid='ed828054caf25401e76eb26a77f844ff9162062e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/souriez/MABe-2025_test-tracking_RAW', endpoint='https://huggingface.co', repo_type='dataset', repo_id='souriez/MABe-2025_test-tracking_RAW'), pr_revision=None, pr_num=None)

In [ ]:
ds_test_tracking.save_to_disk('data/hf/test_tracking/raw')

Saving the dataset (1/1 shards): 100%|██████████| 1089866/1089866 [00:00<00:00, 2190498.82 examples/s]


In [ ]:
def widen(df):
    return df.pivot_table(
    index=['video_frame', 'mouse_id', 'source_filename', 'source_directory', 'source_full_path'],
    columns='bodypart',
    values=['x', 'y']
).reset_index()

In [ ]:
df_test_tracking

,video_frame,mouse_id,bodypart,x,y,source_filename,source_directory,source_full_path
0,0,1,body_center,909.577026,960.265991,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet
1,0,1,ear_left,861.718018,988.614014,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet
2,0,1,ear_right,882.953003,934.760010,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet
3,0,1,headpiece_bottombackright,828.593994,997.226013,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet
4,0,1,headpiece_bottomfrontleft,843.098022,948.182983,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet
...,...,...,...,...,...,...,...,...
1089861,18422,4,lateral_right,794.734009,336.630005,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet
1089862,18422,4,nose,742.864990,430.343994,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet
1089863,18422,4,tail_base,837.304993,326.743988,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet
1089864,18422,4,tail_midpoint,890.411011,261.528015,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet


In [ ]:
df_test_tracking_wide = widen(df_test_tracking)
df_test_tracking_wide

video_frame mouse_id    source_filename source_directory  \
bodypart                                                            
0                  0        1  438887472.parquet   AdaptableSnail   
1                  0        2  438887472.parquet   AdaptableSnail   
2                  0        3  438887472.parquet   AdaptableSnail   
3                  0        4  438887472.parquet   AdaptableSnail   
4                  1        1  438887472.parquet   AdaptableSnail   
...              ...      ...                ...              ...   
73687          18421        4  438887472.parquet   AdaptableSnail   
73688          18422        1  438887472.parquet   AdaptableSnail   
73689          18422        2  438887472.parquet   AdaptableSnail   
73690          18422        3  438887472.parquet   AdaptableSnail   
73691          18422        4  438887472.parquet   AdaptableSnail   

                          source_full_path           x              \
bodypart                                   body_center    ear_left   
0         AdaptableSnail/438887472.parquet  909.577026  861.718018   
1         AdaptableSnail/438887472.parquet  266.998993  261.963013   
2         AdaptableSnail/438887472.parquet  916.135986  970.934021   
3         AdaptableSnail/438887472.parquet  731.554016  716.135986   
4         AdaptableSnail/438887472.parquet  909.872009  862.388000   
...                                    ...         ...         ...   
73687     AdaptableSnail/438887472.parquet  806.585999         NaN   
73688     AdaptableSnail/438887472.parquet  558.085999  576.973022   
73689     AdaptableSnail/438887472.parquet  907.913025  907.481018   
73690     AdaptableSnail/438887472.parquet  645.166016  651.794006   
73691     AdaptableSnail/438887472.parquet  810.341003         NaN   

                                                                         ...  \
bodypart   ear_right headpiece_bottombackleft headpiece_bottombackright  ...   
0         882.953003                      NaN                828.593994  ...   
1         232.800003               276.636993                269.881012  ...   
2         932.093994               980.491028                949.700989  ...   
3         679.950989                      NaN                722.713989  ...   
4         882.604004                      NaN                830.556030  ...   
...              ...                      ...                       ...  ...   
73687     749.017029                      NaN                799.966980  ...   
73688     528.239990               588.831970                597.598022  ...   
73689     950.367981               915.991028                910.966980  ...   
73690     695.297974               656.580994                658.103027  ...   
73691     748.489990                      NaN                800.143982  ...   

                              y                         \
bodypart headpiece_topbackright headpiece_topfrontleft   
0                    995.581970             941.216003   
1                    989.125000             995.487000   
2                    779.706970             819.346985   
3                    201.225998             213.072998   
4                    995.299011             940.520996   
...                         ...                    ...   
73687                408.712006             443.632996   
73688                192.817001                    NaN   
73689                365.627991             352.647003   
73690                638.372009             607.669983   
73691                409.246002             443.308014   

                                                                              \
bodypart headpiece_topfrontright lateral_left lateral_right neck        nose   
0                     942.671021   999.424011    927.291992  NaN  934.216980   
1                     979.388000   909.461975    891.710999  NaN  954.156982   
2                     829.043030   732.607971    797.564026  NaN         NaN   
3   

In [ ]:
df_test_tracking_wide.columns = ['_'.join(col).strip('_') if col[1] else col[0] for col in df_test_tracking_wide.columns.values]
df_test_tracking_wide

,video_frame,mouse_id,source_filename,source_directory,source_full_path,x_body_center,x_ear_left,x_ear_right,x_headpiece_bottombackleft,x_headpiece_bottombackright,...,y_headpiece_topbackright,y_headpiece_topfrontleft,y_headpiece_topfrontright,y_lateral_left,y_lateral_right,y_neck,y_nose,y_tail_base,y_tail_midpoint,y_tail_tip
0,0,1,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet,909.577026,861.718018,882.953003,NaN,828.593994,...,995.581970,941.216003,942.671021,999.424011,927.291992,NaN,934.216980,959.601990,971.898987,NaN
1,0,2,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet,266.998993,261.963013,232.800003,276.636993,269.881012,...,989.125000,995.487000,979.388000,909.461975,891.710999,NaN,954.156982,853.481995,786.799988,748.833008
2,0,3,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet,916.135986,970.934021,932.093994,980.491028,949.700989,...,779.706970,819.346985,829.043030,732.607971,797.564026,NaN,NaN,756.755981,747.757996,726.705017
3,0,4,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet,731.554016,716.135986,679.950989,NaN,722.713989,...,201.225998,213.072998,184.735001,146.319000,107.334999,NaN,190.729996,100.678001,NaN,NaN
4,1,1,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet,909.872009,862.388000,882.604004,NaN,830.556030,...,995.299011,940.520996,941.880981,999.739014,927.427002,NaN,932.940979,959.021973,971.872009,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73687,18421,4,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet,806.585999,NaN,749.017029,NaN,799.966980,...,408.712006,443.632996,409.612000,373.252014,336.451996,NaN,430.928986,326.600006,261.325012,222.235992
73688,18422,1,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet,558.085999,576.973022,528.239990,588.831970,597.598022,...,192.817001,NaN,205.080002,109.327003,110.295998,NaN,175.561005,76.440002,35.533001,59.597000
73689,18422,2,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet,907.913025,907.481018,950.367981,915.991028,910.966980,...,365.627991,352.647003,381.652008,430.450012,447.923004,NaN,378.726990,474.328003,525.737976,594.534973
73690,18422,3,438887472.parquet,AdaptableSnail,AdaptableSnail/438887472.parquet,645.166016,651.794006,695.297974,656.580994,658.103027,...,638.372009,607.669983,637.812988,645.422974,686.749023,NaN,604.960022,694.830994,745.184021,813.388000


In [ ]:
ds_test_tracking_wide = Dataset.from_pandas(df=df_test_tracking_wide)
ds_test_tracking_wide

Dataset({
    features: ['video_frame', 'mouse_id', 'source_filename', 'source_directory', 'source_full_path', 'x_body_center', 'x_ear_left', 'x_ear_right', 'x_headpiece_bottombackleft', 'x_headpiece_bottombackright', 'x_headpiece_bottomfrontleft', 'x_headpiece_bottomfrontright', 'x_headpiece_topbackleft', 'x_headpiece_topbackright', 'x_headpiece_topfrontleft', 'x_headpiece_topfrontright', 'x_lateral_left', 'x_lateral_right', 'x_neck', 'x_nose', 'x_tail_base', 'x_tail_midpoint', 'x_tail_tip', 'y_body_center', 'y_ear_left', 'y_ear_right', 'y_headpiece_bottombackleft', 'y_headpiece_bottombackright', 'y_headpiece_bottomfrontleft', 'y_headpiece_bottomfrontright', 'y_headpiece_topbackleft', 'y_headpiece_topbackright', 'y_headpiece_topfrontleft', 'y_headpiece_topfrontright', 'y_lateral_left', 'y_lateral_right', 'y_neck', 'y_nose', 'y_tail_base', 'y_tail_midpoint', 'y_tail_tip'],
    num_rows: 73692
})

In [ ]:
# ds_test_tracking_wide.push_to_hub('souriez/MABe-2025_test-tracking')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  5.78ba/s]
Processing Files (1 / 1): 100%|██████████| 12.9MB / 12.9MB, 2.92MB/s  
New Data Upload: 100%|██████████| 12.9MB / 12.9MB, 2.92MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:05<00:00,  5.27s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/souriez/MABe-2025_test-tracking/commit/be629afc133c534b39505c95b6321c27a9f57626', commit_message='Upload dataset', commit_description='', oid='be629afc133c534b39505c95b6321c27a9f57626', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/souriez/MABe-2025_test-tracking', endpoint='https://huggingface.co', repo_type='dataset', repo_id='souriez/MABe-2025_test-tracking'), pr_revision=None, pr_num=None)

In [ ]:
ds_test_tracking_wide.save_to_disk('data/MABe-mouse-behavior-detection/hf/test_tracking/')

Saving the dataset (1/1 shards): 100%|██████████| 73692/73692 [00:00<00:00, 382296.89 examples/s]


# train tracking

In [ ]:
df_train_tracking = read_parquet_data_recursive('data/MABe-mouse-behavior-detection/train_tracking')
df_train_tracking

Found 8790 parquet files to process...
Successfully combined 8790 files into dataframe with 800724457 rows


,video_frame,mouse_id,bodypart,x,y,source_filename,source_directory,source_subdirectory,source_full_path
0,0,1,ear_left,0.000000,0.000000,1715161851.parquet,CalMS21_task1,,CalMS21_task1/1715161851.parquet
1,0,1,ear_right,0.000000,0.000000,1715161851.parquet,CalMS21_task1,,CalMS21_task1/1715161851.parquet
2,0,1,hip_left,0.000000,0.000000,1715161851.parquet,CalMS21_task1,,CalMS21_task1/1715161851.parquet
3,0,1,hip_right,0.000000,0.000000,1715161851.parquet,CalMS21_task1,,CalMS21_task1/1715161851.parquet
4,0,1,neck,0.000000,0.000000,1715161851.parquet,CalMS21_task1,,CalMS21_task1/1715161851.parquet
...,...,...,...,...,...,...,...,...,...
800724452,37966,2,hip_left,92.169395,300.044464,1907520217.parquet,TranquilPanther,,TranquilPanther/1907520217.parquet
800724453,37966,2,hip_right,144.066147,314.066406,1907520217.parquet,TranquilPanther,,TranquilPanther/1907520217.parquet
800724454,37966,2,neck,126.047127,273.869965,1907520217.parquet,TranquilPanther,,TranquilPanther/1907520217.parquet
800724455,37966,2,nose,147.939270,253.923950,1907520217.parquet,TranquilPanther,,TranquilPanther/1907520217.parquet


In [ ]:
df_train_tracking.drop("source_subdirectory", axis=1, inplace=True)
df_train_tracking

,video_frame,mouse_id,bodypart,x,y,source_filename,source_directory,source_full_path
0,0,1,ear_left,0.000000,0.000000,1715161851.parquet,CalMS21_task1,CalMS21_task1/1715161851.parquet
1,0,1,ear_right,0.000000,0.000000,1715161851.parquet,CalMS21_task1,CalMS21_task1/1715161851.parquet
2,0,1,hip_left,0.000000,0.000000,1715161851.parquet,CalMS21_task1,CalMS21_task1/1715161851.parquet
3,0,1,hip_right,0.000000,0.000000,1715161851.parquet,CalMS21_task1,CalMS21_task1/1715161851.parquet
4,0,1,neck,0.000000,0.000000,1715161851.parquet,CalMS21_task1,CalMS21_task1/1715161851.parquet
...,...,...,...,...,...,...,...,...
800724452,37966,2,hip_left,92.169395,300.044464,1907520217.parquet,TranquilPanther,TranquilPanther/1907520217.parquet
800724453,37966,2,hip_right,144.066147,314.066406,1907520217.parquet,TranquilPanther,TranquilPanther/1907520217.parquet
800724454,37966,2,neck,126.047127,273.869965,1907520217.parquet,TranquilPanther,TranquilPanther/1907520217.parquet
800724455,37966,2,nose,147.939270,253.923950,1907520217.parquet,TranquilPanther,TranquilPanther/1907520217.parquet


In [ ]:
df_train_tracking_wide = widen(df_train_tracking)

In [ ]:
df_train_tracking_wide.head()

video_frame mouse_id     source_filename  source_directory  \
bodypart                                                              
0                  0        1  1000217804.parquet  MABe22_keypoints   
1                  0        1  1000285113.parquet  MABe22_keypoints   
2                  0        1  1000942438.parquet     MABe22_movies   
3                  0        1  1001006733.parquet  MABe22_keypoints   
4                  0        1  1001081757.parquet  MABe22_keypoints   

                             source_full_path           x                     \
bodypart                                      body_center ear_left ear_right   
0         MABe22_keypoints/1000217804.parquet       334.0    129.0     192.0   
1         MABe22_keypoints/1000285113.parquet       702.0    714.0     650.0   
2            MABe22_movies/1000942438.parquet       439.0     78.0     389.0   
3         MABe22_keypoints/1001006733.parquet       669.0    655.0     644.0   
4         MABe22_keypoints/1001081757.parquet       106.0    514.0     435.0   

                                     ...             y                        \
bodypart forepaw_left forepaw_right  ... lateral_right   neck   nose spine_1   
0               150.0         154.0  ...           NaN  160.0  165.0     NaN   
1               710.0         667.0  ...           NaN  697.0  706.0     NaN   
2                80.0         376.0  ...           NaN  398.0  388.0     NaN   
3               632.0         646.0  ...           NaN  106.0   98.0     NaN   
4               516.0         418.0  ...           NaN  174.0  185.0     NaN   

                                                                               
bodypart spine_2 tail_base tail_middle_1 tail_middle_2 tail_midpoint tail_tip  
0            NaN     147.0           NaN           NaN         178.0    149.0  
1            NaN     701.0           NaN           NaN         179.0    133.0  
2            NaN     402.0           NaN           NaN          35.0    408.0  
3            NaN     112.0           NaN           NaN         124.0     94.0  
4            NaN     162.0           NaN           NaN         380.0    139.0  

[5 rows x 63 columns]

In [ ]:
df_train_tracking_wide.shape

(86075721, 63)

In [ ]:
df_train_tracking_wide.columns = ['_'.join(col).strip('_') if col[1] else col[0] for col in df_train_tracking_wide.columns.values]
df_train_tracking_wide

,video_frame,mouse_id,source_filename,source_directory,source_full_path,x_body_center,x_ear_left,x_ear_right,x_forepaw_left,x_forepaw_right,...,y_lateral_right,y_neck,y_nose,y_spine_1,y_spine_2,y_tail_base,y_tail_middle_1,y_tail_middle_2,y_tail_midpoint,y_tail_tip
0,0,1,1000217804.parquet,MABe22_keypoints,MABe22_keypoints/1000217804.parquet,334.000000,129.000000,192.000000,150.0,154.0,...,NaN,160.0,165.000000,NaN,NaN,147.000000,NaN,NaN,178.0,149.0
1,0,1,1000285113.parquet,MABe22_keypoints,MABe22_keypoints/1000285113.parquet,702.000000,714.000000,650.000000,710.0,667.0,...,NaN,697.0,706.000000,NaN,NaN,701.000000,NaN,NaN,179.0,133.0
2,0,1,1000942438.parquet,MABe22_movies,MABe22_movies/1000942438.parquet,439.000000,78.000000,389.000000,80.0,376.0,...,NaN,398.0,388.000000,NaN,NaN,402.000000,NaN,NaN,35.0,408.0
3,0,1,1001006733.parquet,MABe22_keypoints,MABe22_keypoints/1001006733.parquet,669.000000,655.000000,644.000000,632.0,646.0,...,NaN,106.0,98.000000,NaN,NaN,112.000000,NaN,NaN,124.0,94.0
4,0,1,1001081757.parquet,MABe22_keypoints,MABe22_keypoints/1001081757.parquet,106.000000,514.000000,435.000000,516.0,418.0,...,NaN,174.0,185.000000,NaN,NaN,162.000000,NaN,NaN,380.0,139.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86075716,594027,2,459610814.parquet,BoisterousParrot,BoisterousParrot/459610814.parquet,275.677612,263.746338,267.442230,NaN,NaN,...,NaN,NaN,234.022354,NaN,NaN,246.133057,NaN,NaN,NaN,NaN
86075717,594028,1,1459695188.parquet,BoisterousParrot,BoisterousParrot/1459695188.parquet,273.553986,263.664581,272.940247,NaN,NaN,...,NaN,NaN,246.666824,NaN,NaN,270.969910,NaN,NaN,NaN,NaN
86075718,594028,2,1459695188.parquet,BoisterousParrot,BoisterousParrot/1459695188.parquet,277.249695,266.147308,276.786011,NaN,NaN,...,NaN,NaN,253.542389,NaN,NaN,271.564850,NaN,NaN,NaN,NaN
86075719,594029,1,1459695188.parquet,BoisterousParrot,BoisterousParrot/1459695188.parquet,273.553986,263.646484,273.450806,NaN,NaN,...,NaN,NaN,246.666824,NaN,NaN,270.969910,NaN,NaN,NaN,NaN


In [ ]:
df_train_tracking_wide.to_parquet('data/df_train_tracking_wide.parquet')
df_train_tracking_wide = None

In [ ]:
#reduction size
800_724_457 / 86_075_721

9.302558813303463

In [ ]:
ds_train_tracking_wide = Dataset.from_parquet('data/df_train_tracking_wide.parquet')
ds_train_tracking_wide

Dataset({
    features: ['video_frame', 'mouse_id', 'source_filename', 'source_directory', 'source_full_path', 'x_body_center', 'x_ear_left', 'x_ear_right', 'x_forepaw_left', 'x_forepaw_right', 'x_head', 'x_headpiece_bottombackleft', 'x_headpiece_bottombackright', 'x_headpiece_bottomfrontleft', 'x_headpiece_bottomfrontright', 'x_headpiece_topbackleft', 'x_headpiece_topbackright', 'x_headpiece_topfrontleft', 'x_headpiece_topfrontright', 'x_hindpaw_left', 'x_hindpaw_right', 'x_hip_left', 'x_hip_right', 'x_lateral_left', 'x_lateral_right', 'x_neck', 'x_nose', 'x_spine_1', 'x_spine_2', 'x_tail_base', 'x_tail_middle_1', 'x_tail_middle_2', 'x_tail_midpoint', 'x_tail_tip', 'y_body_center', 'y_ear_left', 'y_ear_right', 'y_forepaw_left', 'y_forepaw_right', 'y_head', 'y_headpiece_bottombackleft', 'y_headpiece_bottombackright', 'y_headpiece_bottomfrontleft', 'y_headpiece_bottomfrontright', 'y_headpiece_topbackleft', 'y_headpiece_topbackright', 'y_headpiece_topfrontleft', 'y_headpiece_topfrontrigh

In [ ]:
ds_train_tracking_wide.column_names

['video_frame',
 'mouse_id',
 'source_filename',
 'source_directory',
 'source_full_path',
 'x_body_center',
 'x_ear_left',
 'x_ear_right',
 'x_forepaw_left',
 'x_forepaw_right',
 'x_head',
 'x_headpiece_bottombackleft',
 'x_headpiece_bottombackright',
 'x_headpiece_bottomfrontleft',
 'x_headpiece_bottomfrontright',
 'x_headpiece_topbackleft',
 'x_headpiece_topbackright',
 'x_headpiece_topfrontleft',
 'x_headpiece_topfrontright',
 'x_hindpaw_left',
 'x_hindpaw_right',
 'x_hip_left',
 'x_hip_right',
 'x_lateral_left',
 'x_lateral_right',
 'x_neck',
 'x_nose',
 'x_spine_1',
 'x_spine_2',
 'x_tail_base',
 'x_tail_middle_1',
 'x_tail_middle_2',
 'x_tail_midpoint',
 'x_tail_tip',
 'y_body_center',
 'y_ear_left',
 'y_ear_right',
 'y_forepaw_left',
 'y_forepaw_right',
 'y_head',
 'y_headpiece_bottombackleft',
 'y_headpiece_bottombackright',
 'y_headpiece_bottomfrontleft',
 'y_headpiece_bottomfrontright',
 'y_headpiece_topbackleft',
 'y_headpiece_topbackright',
 'y_headpiece_topfrontleft',
 'y

In [ ]:
ds_train_tracking_wide.save_to_disk("data/hf/train_tracking/")

Saving the dataset (56/56 shards): 100%|██████████| 86075721/86075721 [06:41<00:00, 214517.21 examples/s]


In [ ]:
# ds_train_tracking_wide.push_to_hub("souriez/MABe-2025_train-tracking")

The file is too large to be uploaded. use this command in the cli after authentication:

```bash
hf upload-large-folder souriez/MABe-2025_train-tracking data/hf/train_tracking --repo-type dataset
```